<a href="https://colab.research.google.com/github/IshratMubarak/House_Price_Prediction_Model/blob/main/House_Price_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# =====================================================================m
# 1. INTERACTIVE FILE UPLOAD
# =====================================================================
print("Please upload your 'HousePricePrediction.csv' file:")
uploaded = files.upload()

file_name = list(uploaded.keys())[0]
df = pd.read_csv(file_name)
print(f"\nOriginal Dataset Shape: {df.shape}")

# =====================================================================
# 2. ADVANCED OUTLIER REMOVAL (Multi-Dimensional)
# =====================================================================
df = df.dropna(subset=['SalePrice'])

# Dynamically clean the most critical driving features to protect linear math
critical_cols = ['SalePrice', 'GrLivArea', 'LotArea', 'TotalBsmtSF']
df_cleaned = df.copy()

for col in critical_cols:
    if col in df_cleaned.columns:
        Q1 = df_cleaned[col].quantile(0.25)
        Q3 = df_cleaned[col].quantile(0.75)
        IQR = Q3 - Q1
        df_cleaned = df_cleaned[(df_cleaned[col] >= Q1 - 1.5 * IQR) &
                                (df_cleaned[col] <= Q3 + 1.5 * IQR)]

print(f"Dataset shape after robust outlier removal: {df_cleaned.shape}")

# =====================================================================
# 3. EXPANDED FEATURE SELECTION (The Engine of Accuracy)
# =====================================================================
y = df_cleaned['SalePrice']

# Added 'OverallQual' and 'GrLivArea' - absolutely mandatory for high accuracy
num_features = ['LotArea', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'TotalBsmtSF', 'GrLivArea']
cat_features = ['MSSubClass', 'MSZoning', 'BldgType', 'Neighborhood']

# Fallback step: verify columns actually exist in your specific CSV file
num_features = [col for col in num_features if col in df_cleaned.columns]
cat_features = [col for col in cat_features if col in df_cleaned.columns]

X = df_cleaned[num_features + cat_features]

# =====================================================================
# 4. TRAIN-TEST SPLIT
# =====================================================================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# =====================================================================
# 5. PREPROCESSING PIPELINE
# =====================================================================
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_features),
        ('cat', categorical_transformer, cat_features)
    ])

# =====================================================================
# 6. HYPERPARAMETER TUNING (GridSearchCV for Exceptional Results)
# =====================================================================
print("\nOptimizing Hyperparameters via Grid Search (This may take a minute)...")
print("-" * 50)

# Base pipelines
rf_base = Pipeline([('preprocessor', preprocessor), ('regressor', RandomForestRegressor(random_state=42))])
svm_base = Pipeline([('preprocessor', preprocessor), ('regressor', SVR(kernel='rbf'))])

# Parameter grids to test
rf_param_grid = {
    'regressor__n_estimators': [100, 200],
    'regressor__max_depth': [10, 15, None],
    'regressor__min_samples_split': [2, 5]
}

svm_param_grid = {
    'regressor__C': [50000, 100000, 200000],
    'regressor__epsilon': [0.1, 1, 10]
}

# Execute searches
rf_grid = GridSearchCV(rf_base, rf_param_grid, cv=3, scoring='r2', n_jobs=-1).fit(X_train, y_train)
svm_grid = GridSearchCV(svm_base, svm_param_grid, cv=3, scoring='r2', n_jobs=-1).fit(X_train, y_train)

# =====================================================================
# 7. EVALUATION
# =====================================================================
models = {
    "Tuned Random Forest": rf_grid.best_estimator_,
    "Tuned SVM Regressor": svm_grid.best_estimator_
}

predictions = {}
best_model_name = None
best_r2 = -float('inf')

for name, pipeline in models.items():
    y_pred = pipeline.predict(X_test)
    predictions[name] = y_pred

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    print(f"Model: {name}")
    print(f"  Best Parameters          : {rf_grid.best_params_ if 'Forest' in name else svm_grid.best_params_}")
    print(f"  Mean Absolute Error (MAE): ${mae:,.2f}")
    print(f"  Root Mean Squared Error  : ${rmse:,.2f}")
    print(f"  R-squared Score (R²)     : {r2:.4f}")
    print("-" * 50)

    if r2 > best_r2:
        best_r2 = r2
        best_model_name = name

# =====================================================================
# 8. VISUALIZE PERFORMANCE
# =====================================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
colors = ['purple', 'teal']
for i, (name, y_pred) in enumerate(predictions.items()):
    sns.scatterplot(x=y_test, y=y_pred, alpha=0.6, color=colors[i], ax=axes[i])
    axes[i].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
    axes[i].set_title(f'{name}\nActual vs Predicted', fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Actual Sale Price ($)')
    if i == 0: axes[i].set_ylabel('Predicted Sale Price ($)')

plt.tight_layout()
plot_filename = 'Optimized_Model_Performance.png'
plt.savefig(plot_filename, dpi=300)
plt.show()

# =====================================================================
# 9. INTERACTIVE PREDICTOR
# =====================================================================
print("\n" + "="*50)
print("             HOUSE PRICE PREDICTOR (OPTIMIZED)")
print("="*50)
user_choice = input("Would you like to predict a house price? (yes/no): ").strip().lower()

if user_choice == 'yes':
    print(f"\nUsing the best optimized model: {best_model_name}")
    chosen_pipeline = models[best_model_name]
    input_df = pd.DataFrame(columns=X.columns)

    for col in num_features: input_df.loc[0, col] = df_cleaned[col].median()
    for col in cat_features: input_df.loc[0, col] = df_cleaned[col].mode()[0]

    print("\nPlease enter values below. Press [ENTER] to use the baseline default value.\n")

    for col in num_features:
        val = input(f"{col} (Default/Median: {input_df.loc[0, col]}): ").strip()
        if val != "": input_df.loc[0, col] = float(val)

    for col in cat_features:
        val = input(f"{col} (Default/Mode: '{input_df.loc[0, col]}'): ").strip()
        if val != "":
            try:
                if type(X[col].iloc[0]) != str: input_df.loc[0, col] = type(X[col].iloc[0])(val)
                else: input_df.loc[0, col] = val
            except ValueError: input_df.loc[0, col] = val

    predicted_val = chosen_pipeline.predict(input_df)[0]
    print("\n" + "-"*35 + "\n         PREDICTION RESULT\n" + "-"*35)
    print(f"Estimated Market Value: ${predicted_val:,.2f}\n" + "-"*35)

# Save data and trigger downloads
cleaned_filename = "Optimized_HousePrices.csv"
df_cleaned.to_csv(cleaned_filename, index=False)
files.download(cleaned_filename)
files.download(plot_filename)

Please upload your 'HousePricePrediction.csv' file:
